In [65]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

_____
# ARPEGE

In [109]:
def agescmed(z, U10,f, K, a, omega, dxS):
    # cstes
    rho = 1.2
    #norm
    u10 = U10**2
    k= K**2
    
    Cd = (2.7/u10 + 0.142 +0.076*u10)/1e3 # neutral drag coeff
    Az = 1.2*1e-4 * u10 # vertical velocity profile considered uniform in the whole bassin m2.s-1
    tau = rho * Cd * u10 * U10 # wind stress
    m=(1+j)*np.sqrt(f/(2*Az))


    Us0 = a**2 * omega *k * np.exp(2*k*z)*K
    
    Ue = tau/(rho*Az*m)*np.exp(m*z)
    Utaus = dxS/(rho*Az*m)*np.exp(m*z)
    Us = m**2*Us0/(4*k**2-m**2)*np.exp(2*k*z)
    Ues = -2*k*m*Us0/(4*k**2-m**2)*np.exp(m*z)

    return Ue + Utaus + Us + Ues
    
    

SyntaxError: invalid syntax (2011828709.py, line 14)

In [79]:
def cst_rio_z0(df):
    
    theta0 = - 22.5 * np.pi / 180 * np.sign(df.f)
    beta0 = 0.6
    
    uek_e = beta0 * (np.cos(theta0) * df.taue - np.sin(theta0) * df.taun)
    uek_n = beta0 * (np.sin(theta0) * df.taue + np.cos(theta0) * df.taun)
    df['wde0'] = df.f * uek_n
    df['wdn0'] = -df.f * uek_e

def cst_rio_z15(df):
    
    theta15 = - 37.5 * np.pi / 180 * np.sign(df.f)#adapted for mediterranean sea
    beta15 = 0.15
    
    uek_e = beta15 * (np.cos(theta15) * df.taue - np.sin(theta15) * df.taun)
    uek_n = beta15 * (np.sin(theta15) * df.taue + np.cos(theta15) * df.taun)
    df['wde15'] = df.f * uek_n
    df['wdn15'] = -df.f * uek_e

In [98]:
arpege = glob('/Users/mdemol/DATA_WIND/arpege_hr/*.nc')
era5  =glob('/Users/mdemol/DATA_WIND/era5/*.nc')

In [94]:
def stress_from_velocity(ds):
    """ 
    taue = rho * Cd * U *u with Cd = 0.008 +0.008*U10
    see  https://confluence.ecmwf.int/display/CKB/ERA5%3A+data+documentation#ERA5:datadocumentation-Instantaneousparameters in The instantaneous turbulent surface stress components (eastward and northward) and friction velocity tend to be too small
    """
    
    rho = 1.204 #20C
    Cd = 0.0015 #lambda U : (0.008 + 0.0008* U)*1e-3
    U = np.sqrt(ds['u10m']**2+ds['v10m']**2)
    #print(Cd(U))
    ds['taue'] = rho * Cd *U * ds['u10m'] #*1e6
    ds['taun'] = rho * Cd *U * ds['v10m'] #*1e6



In [97]:
dsa = xr.open_dataset(arpege[0])
stress_from_velocity(dsa)
dsa['f'] = 2 * 2 * np.pi / 86164.1 * np.sin(dsa.latitude * np.pi / 180)
cst_rio_z15(dsa)
dsa.taue.max(), dsa.taue.min()

(<xarray.DataArray 'taue' ()> Size: 8B
 array(0.2413148)
 Coordinates:
     height   float32 4B ...,
 <xarray.DataArray 'taue' ()> Size: 8B
 array(-0.25409555)
 Coordinates:
     height   float32 4B ...)

In [105]:
dse = xr.open_dataset(era5[0])
dse

<xarray.Dataset> Size: 828MB
Dimensions:    (longitude: 61, latitude: 33, time: 3672)
Coordinates:
  * longitude  (longitude) float32 244B 0.0 0.25 0.5 0.75 ... 14.5 14.75 15.0
  * latitude   (latitude) float32 132B 44.0 43.75 43.5 43.25 ... 36.5 36.25 36.0
  * time       (time) datetime64[ns] 29kB 2023-03-01 ... 2023-07-31T23:00:00
Data variables: (12/14)
    u10        (time, latitude, longitude) float64 59MB ...
    v10        (time, latitude, longitude) float64 59MB ...
    t2m        (time, latitude, longitude) float64 59MB ...
    ewss       (time, latitude, longitude) float64 59MB ...
    iews       (time, latitude, longitude) float64 59MB ...
    inss       (time, latitude, longitude) float64 59MB ...
    ...         ...
    sst        (time, latitude, longitude) float64 59MB ...
    ssr        (time, latitude, longitude) float64 59MB ...
    ssrc       (time, latitude, longitude) float64 59MB ...
    str        (time, latitude, longitude) float64 59MB ...
    strc       (time, latitude, longitude) float64 59MB ...
    sp         (time, latitude, longitude) float64 59MB ...
Attributes:
    Conventions:  CF-1.6
    history:      2024-09-10 21:05:21 GMT by grib_to_netcdf-2.28.1: /opt/ecmw...

In [106]:
abs(dsa.taue).median(), abs(dse.iews).median()

(<xarray.DataArray 'taue' ()> Size: 8B
 array(0.00758888)
 Coordinates:
     height   float32 4B ...,
 <xarray.DataArray 'iews' ()> Size: 8B
 array(0.0272489))

_____
# Compare ERA5 vs arpege

In [110]:
dsa

<xarray.Dataset> Size: 35MB
Dimensions:    (time: 24, latitude: 181, longitude: 201)
Coordinates:
  * time       (time) datetime64[ns] 192B 2023-08-09 ... 2023-08-09T23:00:00
  * latitude   (latitude) float32 724B 30.0 30.1 30.2 30.3 ... 47.8 47.9 48.0
  * longitude  (longitude) float32 804B -1.0 -0.9 -0.8 -0.7 ... 18.8 18.9 19.0
    height     float32 4B ...
Data variables:
    hu2m       (time, latitude, longitude) float32 3MB ...
    nebul      (time, latitude, longitude) float32 3MB ...
    pmer       (time, latitude, longitude) float32 3MB ...
    t2m        (time, latitude, longitude) float32 3MB ...
    u10m       (time, latitude, longitude) float32 3MB -5.328 -5.188 ... 1.95
    v10m       (time, latitude, longitude) float32 3MB -1.015 ... -0.05846
    taue       (time, latitude, longitude) float32 3MB -0.05606 ... 0.007379
    taun       (time, latitude, longitude) float32 3MB -0.01068 ... -0.0002212
    f          (latitude) float32 724B 7.292e-05 7.314e-05 ... 0.0001084
    wde15      (latitude, time, longitude) float32 3MB 2.806e-07 ... -7.588e-08
    wdn15      (latitude, time, longitude) float32 3MB 5.576e-07 ... -9.298e-08
Attributes: (12/40)
    data_type:               OCO straight grid
    format_version:          1.2
    title:                   Meteo-France Arpege forecast and analysis
    Conventions:             CF-1.3
    netcdf_version:          3.5
    product_name:            METEOFRANCE_ARPEGE-HR_20230809T00Z.nc
    ...                      ...
    data_centre:             CD-OCO
    data_centre_references:  http://www.previmer.org/
    contact:                 cdoco-exploit@ifremer.fr
    distribution_statement:  Data restrictions: for registered users only
    operational_status:      operational
    quality_index:           1